In [ ]:
# @title
# ==========================================================
# SISTEMA DE ANÁLISE DE VENDAS
# SQLite + Pandas + Matplotlib + Seaborn
# ==========================================================


# ==========================================================
# PASSO 1 - CONECTAR AO BANCO DE DADOS E CRIAR A TABELA
# ==========================================================

import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


# Conecta ao banco de dados SQLite
conexao = sqlite3.connect('dados_vendas.db')


# Cria o cursor
cursor = conexao.cursor()


# Cria a tabela caso ela ainda não exista
cursor.execute('''
CREATE TABLE IF NOT EXISTS vendas1 (
    id_venda INTEGER PRIMARY KEY AUTOINCREMENT,
    data_venda DATE,
    produto TEXT,
    categoria TEXT,
    valor_venda REAL
)
''')


# Insere os dados de vendas
cursor.execute('''
INSERT INTO vendas1
(data_venda, produto, categoria, valor_venda)
VALUES
('2023-01-01', 'Produto A', 'Eletrônicos', 1500.00),
('2023-01-05', 'Produto B', 'Roupas', 350.00),
('2023-02-10', 'Produto C', 'Eletrônicos', 1200.00),
('2023-03-15', 'Produto D', 'Livros', 200.00),
('2023-03-20', 'Produto E', 'Eletrônicos', 800.00),
('2023-04-02', 'Produto F', 'Roupas', 400.00),
('2023-05-05', 'Produto G', 'Livros', 150.00),
('2023-06-10', 'Produto H', 'Eletrônicos', 1000.00),
('2023-07-20', 'Produto I', 'Roupas', 600.00),
('2023-08-25', 'Produto J', 'Eletrônicos', 700.00),
('2023-09-30', 'Produto K', 'Livros', 300.00),
('2023-10-05', 'Produto L', 'Roupas', 450.00),
('2023-11-15', 'Produto M', 'Eletrônicos', 900.00),
('2023-12-20', 'Produto N', 'Livros', 250.00)
''')


# Confirma as alterações no banco
conexao.commit()


# ==========================================================
# PASSO 2 - EXPLORAR E PREPARAR OS DADOS
# ==========================================================


# Lê os dados da tabela usando Pandas
df_vendas = pd.read_sql_query(
    'SELECT * FROM vendas1',
    conexao
)


# Mostra os primeiros registros
print("\n========== PRIMEIROS REGISTROS ==========")
print(df_vendas.head())


# Mostra informações sobre o DataFrame
print("\n========== INFORMAÇÕES ==========")
print(df_vendas.info())


# Mostra a quantidade de linhas e colunas
print("\n========== TAMANHO DO DATAFRAME ==========")
print("Linhas:", df_vendas.shape[0])
print("Colunas:", df_vendas.shape[1])


# Verifica se existem valores nulos
print("\n========== VALORES NULOS ==========")
print(df_vendas.isnull().sum())


# Converte a coluna de data para o formato de data do Pandas
df_vendas['data_venda'] = pd.to_datetime(
    df_vendas['data_venda']
)


# Cria uma coluna com o mês da venda
df_vendas['mes'] = df_vendas['data_venda'].dt.month


# Cria uma coluna com o nome do mês
df_vendas['nome_mes'] = df_vendas['data_venda'].dt.month_name()


# ==========================================================
# PASSO 3 - ANÁLISE DOS DADOS
# ==========================================================


# ----------------------------------------------------------
# 3.1 - Faturamento total
# ----------------------------------------------------------

faturamento_total = df_vendas['valor_venda'].sum()

print("\n========== FATURAMENTO TOTAL ==========")
print(f"R$ {faturamento_total:.2f}")


# ----------------------------------------------------------
# 3.2 - Média das vendas
# ----------------------------------------------------------

media_vendas = df_vendas['valor_venda'].mean()

print("\n========== MÉDIA DAS VENDAS ==========")
print(f"R$ {media_vendas:.2f}")


# ----------------------------------------------------------
# 3.3 - Maior venda
# ----------------------------------------------------------

maior_venda = df_vendas['valor_venda'].max()

print("\n========== MAIOR VENDA ==========")
print(f"R$ {maior_venda:.2f}")


# ----------------------------------------------------------
# 3.4 - Menor venda
# ----------------------------------------------------------

menor_venda = df_vendas['valor_venda'].min()

print("\n========== MENOR VENDA ==========")
print(f"R$ {menor_venda:.2f}")


# ----------------------------------------------------------
# 3.5 - Faturamento por categoria
# ----------------------------------------------------------

vendas_categoria = df_vendas.groupby(
    'categoria'
)['valor_venda'].sum().sort_values(
    ascending=False
)


print("\n========== VENDAS POR CATEGORIA ==========")
print(vendas_categoria)


# ----------------------------------------------------------
# 3.6 - Produto com maior venda
# ----------------------------------------------------------

produto_maior_venda = df_vendas.loc[
    df_vendas['valor_venda'].idxmax()
]


print("\n========== PRODUTO COM MAIOR VENDA ==========")
print("Produto:", produto_maior_venda['produto'])
print("Categoria:", produto_maior_venda['categoria'])
print(f"Valor: R$ {produto_maior_venda['valor_venda']:.2f}")


# ----------------------------------------------------------
# 3.7 - Faturamento por mês
# ----------------------------------------------------------

vendas_mes = df_vendas.groupby(
    'mes'
)['valor_venda'].sum()


print("\n========== VENDAS POR MÊS ==========")
print(vendas_mes)


# ----------------------------------------------------------
# 3.8 - Quantidade de vendas por categoria
# ----------------------------------------------------------

quantidade_categoria = df_vendas[
    'categoria'
].value_counts()


print("\n========== QUANTIDADE DE VENDAS POR CATEGORIA ==========")
print(quantidade_categoria)


# ==========================================================
# PASSO 4 - VISUALIZAÇÃO DOS DADOS
# ==========================================================


# Define o estilo dos gráficos
sns.set(style='whitegrid')


# ----------------------------------------------------------
# 4.1 - Gráfico de vendas por categoria
# ----------------------------------------------------------

plt.figure(figsize=(8, 5))

sns.barplot(
    data=df_vendas,
    x='categoria',
    y='valor_venda',
    estimator=sum
)

plt.title('Faturamento por Categoria')
plt.xlabel('Categoria')
plt.ylabel('Faturamento (R$)')

plt.tight_layout()
plt.show()


# ----------------------------------------------------------
# 4.2 - Gráfico das vendas ao longo dos meses
# ----------------------------------------------------------

plt.figure(figsize=(10, 5))

sns.barplot(
    data=df_vendas,
    x='mes',
    y='valor_venda',
    estimator=sum
)

plt.title('Faturamento por Mês')
plt.xlabel('Mês')
plt.ylabel('Faturamento (R$)')

plt.tight_layout()
plt.show()


# ----------------------------------------------------------
# 4.3 - Gráfico dos produtos
# ----------------------------------------------------------

plt.figure(figsize=(12, 5))

sns.barplot(
    data=df_vendas,
    x='produto',
    y='valor_venda'
)

plt.title('Valor de Venda por Produto')
plt.xlabel('Produto')
plt.ylabel('Valor da Venda (R$)')

plt.xticks(rotation=45)

plt.tight_layout()
plt.show()


# ----------------------------------------------------------
# 4.4 - Distribuição dos valores de venda
# ----------------------------------------------------------

plt.figure(figsize=(8, 5))

sns.histplot(
    data=df_vendas,
    x='valor_venda',
    bins=6,
    kde=True
)

plt.title('Distribuição dos Valores de Venda')
plt.xlabel('Valor da Venda (R$)')
plt.ylabel('Quantidade')

plt.tight_layout()
plt.show()


# ==========================================================
# PASSO 5 - CONCLUSÃO E INSIGHTS
# ==========================================================


print("\n")
print("==========================================================")
print("                 CONCLUSÃO DA ANÁLISE")
print("==========================================================")


print(f"""
O faturamento total registrado foi de R$ {faturamento_total:.2f}.

A média das vendas foi de R$ {media_vendas:.2f}.

A maior venda registrada foi de R$ {maior_venda:.2f}.

A menor venda registrada foi de R$ {menor_venda:.2f}.

A categoria com maior faturamento foi:
{vendas_categoria.index[0]}

Essa análise permite identificar quais categorias e períodos
geraram maior faturamento.

A empresa pode utilizar essas informações para:

- acompanhar o desempenho das categorias;
- identificar períodos de maior faturamento;
- analisar quais produtos possuem maior valor de venda;
- planejar campanhas e promoções;
- melhorar o planejamento de estoque;
- acompanhar a evolução das vendas ao longo do tempo.
""")


# Fecha a conexão com o banco
conexao.close()

print("Análise finalizada!")